# ANON AI ULTRA — QLoRA sohbet uyarlaması (Qwen3 1.7B)

Bu çalışma eski 30M sıfırdan modeli silmez veya değiştirmez. Önceden eğitilmiş Apache-2.0 lisanslı `Qwen/Qwen3-1.7B` modelini 4-bit yükler ve Türkçe sohbet verisiyle LoRA adaptörü eğitir. Temel modelin hazır sohbet yeteneği korunur; çıktı önce Colab'da test edilir.

Ücretsiz Colab GPU belleği ve çalışma süresi garanti değildir. GPU T4 önerilir. Adaptör tek başına tam model değildir.

In [ ]:
import os, sys, subprocess, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU bulunamadı')
assert torch.cuda.is_available(), 'Colab > Runtime > Change runtime type > GPU seç.'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'transformers', 'datasets', 'accelerate', 'peft', 'bitsandbytes', 'safetensors'], check=True)
print('Kurulum tamamlandı.')

## 1. Depoyu ve Türkçe sohbet verisini hazırla

Önceki 30M model ZIP dosyasına bu eğitimde ihtiyaç yok.

In [ ]:
import os, subprocess, sys
REPO_DIR = '/content/ANON-AI-ULTRA'
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git','clone','--depth','1','https://github.com/furkanmeyra2173-create/ANON-AI-ULTRA.git',REPO_DIR],check=True)
else:
    subprocess.run(['git','-C',REPO_DIR,'pull','--ff-only'],check=True)
os.chdir(REPO_DIR)
DATA_PATH = '/content/turkish_chat_train.jsonl'
cmd = [sys.executable,'training/prepare_instruction_data.py','--dataset-id','cisimcik/turkish-chat-max-25k','--config','all','--split','train','--language-mode','tr_only','--prompt-language','tr','--max-samples','20000','--output',DATA_PATH]
subprocess.run(cmd,check=True)
print('Veri satırı:', sum(1 for _ in open(DATA_PATH,encoding='utf-8')))

## 2. Qwen3 1.7B modelini 4-bit yükle

Model kartı: https://huggingface.co/Qwen/Qwen3-1.7B — Apache-2.0 lisanslıdır; yine de veri seti ve model koşullarını yayımdan önce ayrıca kontrol et.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
MODEL_ID = 'Qwen/Qwen3-1.7B'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'
quant_config = BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_compute_dtype=torch.float16,bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID,quantization_config=quant_config,device_map='auto',torch_dtype=torch.float16,trust_remote_code=True)
model.config.use_cache = False
print('Model yüklendi:', MODEL_ID)
print('Tokenizer:', len(tokenizer), '| GPU:', torch.cuda.get_device_name(0))

## 3. Önce temel modeli dene

Bu adım, eğitimden önce modelin Türkçe cevap verebildiğini kontrol eder.

In [ ]:
def ask_model(question, max_new_tokens=120):
    messages = [{'role':'user','content':question}]
    prompt = tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True,enable_thinking=False)
    inputs = tokenizer(prompt,return_tensors='pt').to(model.device)
    with torch.inference_mode():
        output = model.generate(**inputs,max_new_tokens=max_new_tokens,do_sample=False,pad_token_id=tokenizer.pad_token_id,eos_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(output[0][inputs['input_ids'].shape[1]:],skip_special_tokens=True).strip()
for q in ['Merhaba, nasılsın?','Türkiye’nin başkenti neresidir?','Yapay zekâ nedir?']:
    print('\nSoru:',q,'\nCevap:',ask_model(q))

## 4. Veriyi Qwen sohbet biçimine dönüştür

Bu aşama mevcut veri kümesindeki `system`, `user`, `assistant` mesajlarını Qwen'in kendi sohbet şablonuna çevirir.

In [ ]:
import json
from datasets import Dataset
records=[]
with open(DATA_PATH,encoding='utf-8') as f:
    for line in f:
        row=json.loads(line)
        msgs=row.get('messages',[])
        if any(m.get('role')=='user' for m in msgs) and any(m.get('role')=='assistant' for m in msgs):
            records.append({'messages':msgs})
assert len(records)>=100, f'Yeterli sohbet verisi yok: {len(records)}'
raw=Dataset.from_list(records).shuffle(seed=42)
def render_batch(batch):
    texts=[]
    for msgs in batch['messages']:
        clean=[{'role':m['role'],'content':m['content']} for m in msgs if m.get('role') in ('system','user','assistant') and isinstance(m.get('content'),str) and m['content'].strip()]
        texts.append(tokenizer.apply_chat_template(clean,tokenize=False,add_generation_prompt=False,enable_thinking=False))
    return {'text':texts}
rendered=raw.map(render_batch,batched=True,remove_columns=raw.column_names)
split=rendered.train_test_split(test_size=0.05,seed=42)
print('Eğitim:',len(split['train']),'| Doğrulama:',len(split['test']))
print('Örnek sohbet:',split['train'][0]['text'][:500])

## 5. LoRA adaptörünü eğit

Başlangıç ayarları T4 GPU için temkinlidir. Eğitim yalnızca adaptör parametrelerini günceller; temel model dosyası değiştirilmez.

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
model=prepare_model_for_kbit_training(model)
lora_config=LoraConfig(r=16,lora_alpha=32,lora_dropout=0.05,bias='none',task_type='CAUSAL_LM',target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
model=get_peft_model(model,lora_config)
model.print_trainable_parameters()
MAX_LENGTH=1024
def tokenize_batch(batch):
    return tokenizer(batch['text'],truncation=True,max_length=MAX_LENGTH,padding=False)
tokenized=split.map(tokenize_batch,batched=True,remove_columns=['text'])
collator=DataCollatorForLanguageModeling(tokenizer=tokenizer,mlm=False)
args=TrainingArguments(output_dir='/content/anon-qwen3-lora-checkpoints',per_device_train_batch_size=1,per_device_eval_batch_size=1,gradient_accumulation_steps=8,num_train_epochs=1,learning_rate=1e-4,warmup_ratio=0.03,lr_scheduler_type='cosine',logging_steps=10,eval_strategy='steps',eval_steps=100,save_strategy='steps',save_steps=100,save_total_limit=2,fp16=True,bf16=False,gradient_checkpointing=True,optim='paged_adamw_8bit',report_to='none',dataloader_num_workers=0)
trainer=Trainer(model=model,args=args,train_dataset=tokenized['train'],eval_dataset=tokenized['test'],data_collator=collator,processing_class=tokenizer)
train_result=trainer.train()
ADAPTER_DIR='/content/ANON-AI-ULTRA-Qwen3-1.7B-LoRA'
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print('LoRA adaptörü kaydedildi:',ADAPTER_DIR)
print('Eğitim metrikleri:',train_result.metrics)

## 6. Eğitilmiş adaptörü test et

Testte Qwen'in orijinal sohbet şablonu kullanılır. Cevapları eski 30M modelin çıktılarıyla karşılaştır.

In [ ]:
from peft import PeftModel
model.eval()
for q in ['Merhaba, nasılsın?','Türkiye’nin başkenti neresidir?','Yapay zekâ nedir?','Almanca merhaba nasıl söylenir?','Bana kendini kısaca tanıtır mısın?']:
    print('\nSORU:',q,'\nANON AI:',ask_model(q,160))

## 7. Adaptörü ZIP olarak indir

Bu ZIP yalnızca LoRA adaptörüdür. Tek başına uygulamaya yüklenemez; dağıtım öncesinde temel modelle birleştirme ve uygun biçime dönüştürme adımı gerekir.

In [ ]:
import shutil, os
ZIP_PATH=shutil.make_archive('/content/ANON-AI-ULTRA-Qwen3-1.7B-LoRA','zip',ADAPTER_DIR)
print('Hazır:',ZIP_PATH,'Boyut MB:',round(os.path.getsize(ZIP_PATH)/1024/1024,2))
from google.colab import files
files.download(ZIP_PATH)